<a href="https://colab.research.google.com/github/james-mawuena/student-performance-prediction/blob/main/student_performance_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression

In [1]:
import pandas as pd

file = "Dataset on the academic performance of students in 12 programs from a private university.xlsx"
df = pd.read_excel(file, sheet_name="Sheet1")
df = df.iloc[:, :10]
df = df.rename(columns={"SGPA": "SecondaryGrade"})

print(df.shape)
df.head()

(3046, 10)


,ID No,Prog Code,Gender,YoG,CGPA,CGPA100,CGPA200,CGPA300,CGPA400,SecondaryGrade
0,42308,ICE,Female,2010,3.227513,2.875000,3.475000,2.615385,2.898305,3.125000
1,70978,BCH,Female,2010,3.576271,3.250000,4.261905,3.368421,3.469388,3.020833
2,31602,BCH,Male,2010,2.211454,1.777778,1.979167,1.489583,2.511111,2.187500
3,63847,BCH,Male,2010,2.702970,2.673913,2.442308,2.000000,2.348315,3.194444
4,30158,BCH,Female,2010,3.881657,3.608696,3.687500,3.625000,4.581395,4.236111


In [2]:
X = df[["CGPA100", "SecondaryGrade", "Prog Code"]]
X = pd.get_dummies(X, columns=["Prog Code"])

y = df["CGPA"]

print(X.shape)
print(y.head())

(3046, 19)
0    3.227513
1    3.576271
2    2.211454
3    2.702970
4    3.881657
Name: CGPA, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training students:", X_train.shape[0])
print("Test students:", X_test.shape[0])

Training students: 2436
Test students: 610


In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

compare = pd.DataFrame({
    "Real": y_test.values,
    "Guess": predictions.round(2)
})
compare.head(8)

,Real,Guess
0,4.403727,4.02
1,4.443114,4.01
2,3.450000,3.11
3,3.521472,3.51
4,2.314286,2.91
5,4.359091,4.28
6,2.698413,3.82
7,3.871111,2.90


In [5]:
from sklearn.metrics import mean_absolute_error

lazy_guess = [y_train.mean()] * len(y_test)

print("Lazy guess error:", round(mean_absolute_error(y_test, lazy_guess), 3))
print("Model error:", round(mean_absolute_error(y_test, predictions), 3))

Lazy guess error: 0.588
Model error: 0.314


In [6]:
import joblib
from google.colab import files

joblib.dump(model, "model.pkl")
joblib.dump(list(X.columns), "columns.pkl")

files.download("model.pkl")
files.download("columns.pkl")

print(list(X.columns))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['CGPA100', 'SecondaryGrade', 'Prog Code_BCH', 'Prog Code_BLD', 'Prog Code_CEN', 'Prog Code_CHE', 'Prog Code_CHM', 'Prog Code_CIS', 'Prog Code_CVE', 'Prog Code_EEE', 'Prog Code_ICE', 'Prog Code_MAT', 'Prog Code_MCB', 'Prog Code_MCE', 'Prog Code_MIS', 'Prog Code_PET', 'Prog Code_PHYE', 'Prog Code_PHYG', 'Prog Code_PHYR']


# Logistic Regression

Two Categories

In [7]:
y_class = (df["CGPA"] >= 3.5).astype(int)

X_train, X_test, yc_train, yc_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)

print("Good degree in training group:", round(yc_train.mean(), 2))
print("Good degree in test group:", round(yc_test.mean(), 2))

Good degree in training group: 0.54
Good degree in test group: 0.5


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, yc_train)

chance = clf.predict_proba(X_test)[:, 1]
guess = clf.predict(X_test)

lazy = [1] * len(yc_test)
print("Lazy guess accuracy:", round(accuracy_score(yc_test, lazy), 3))
print("Model accuracy:", round(accuracy_score(yc_test, guess), 3))

compare = pd.DataFrame({
    "Real": yc_test.values,
    "Chance": chance.round(2),
    "Guess": guess
})
compare.head(8)

Lazy guess accuracy: 0.502
Model accuracy: 0.83


,Real,Chance,Guess
0,1,0.93,1
1,1,0.92,1
2,0,0.19,0
3,1,0.57,1
4,0,0.08,0
5,1,0.97,1
6,0,0.84,1
7,1,0.07,0


In [9]:
import joblib
from google.colab import files

joblib.dump(clf, "good_degree_model.pkl")
files.download("good_degree_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
check = pd.DataFrame({
    "Gender": df.loc[X_test.index, "Gender"].values,
    "Real CGPA": y_test.values,
    "Guess CGPA": predictions,
    "Real good": yc_test.values,
    "Guess good": guess,
})

check["Miss size"] = (check["Guess CGPA"] - check["Real CGPA"]).abs()
check["Miss direction"] = check["Guess CGPA"] - check["Real CGPA"]
check["Right"] = (check["Real good"] == check["Guess good"])

summary = check.groupby("Gender").agg(
    Students=("Gender", "count"),
    MAE=("Miss size", "mean"),
    Avg_miss_direction=("Miss direction", "mean"),
    Accuracy=("Right", "mean"),
    Really_good=("Real good", "mean"),
    Model_said_good=("Guess good", "mean"),
).round(3)

summary

,Students,MAE,Avg_miss_direction,Accuracy,Really_good,Model_said_good
Gender,,,,,,
Female,215,0.305,-0.137,0.860,0.637,0.544
Male,395,0.319,0.123,0.813,0.428,0.484


Four Cetegories

In [11]:
bins = [0, 2.4, 3.5, 4.5, 5.01]
names = ["Third", "Second Lower", "Second Upper", "First"]

y_four = pd.cut(df["CGPA"], bins=bins, labels=names, right=False)

X_train, X_test, y4_train, y4_test = train_test_split(
    X, y_four, test_size=0.2, random_state=42
)

print("Training students per class:")
print(y4_train.value_counts())
print()
print("Share of each class in the test group:")
print(y4_test.value_counts(normalize=True).round(2))

Training students per class:
CGPA
Second Upper    1130
Second Lower     957
First            184
Third            165
Name: count, dtype: int64

Share of each class in the test group:
CGPA
Second Upper    0.43
Second Lower    0.42
Third           0.08
First           0.07
Name: proportion, dtype: float64


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf4 = LogisticRegression(max_iter=1000)
clf4.fit(X_train, y4_train)

chances = clf4.predict_proba(X_test)
picked = clf4.predict(X_test)

lazy = ["Second Upper"] * len(y4_test)
print("Lazy guess accuracy:", round(accuracy_score(y4_test, lazy), 3))
print("Model accuracy:", round(accuracy_score(y4_test, picked), 3))

table = pd.DataFrame(chances.round(2), columns=clf4.classes_)
table["Real"] = y4_test.values
table["Picked"] = picked
table.head(5)

Lazy guess accuracy: 0.433
Model accuracy: 0.721


,First,Second Lower,Second Upper,Third,Real,Picked
0,0.06,0.08,0.85,0.00,Second Upper,Second Upper
1,0.02,0.10,0.88,0.00,Second Upper,Second Upper
2,0.00,0.71,0.20,0.09,Second Lower,Second Lower
3,0.00,0.43,0.56,0.02,Second Upper,Second Upper
4,0.00,0.74,0.09,0.17,Third,Second Lower


In [13]:
pd.crosstab(
    y4_test.values, picked,
    rownames=["Real"], colnames=["Picked"]
).reindex(index=names, columns=names).fillna(0).astype(int)

Picked,Third,Second Lower,Second Upper,First
Real,,,,
Third,9,38,0,0
Second Lower,4,197,56,0
Second Upper,0,48,207,9
First,0,0,15,27


In [14]:
from sklearn.metrics import recall_score

recall = recall_score(y4_test, picked, labels=names, average=None)

pd.DataFrame({
    "Class": names,
    "Real students": y4_test.value_counts().reindex(names).values,
    "Recall (%)": (recall * 100).round(0)
})

,Class,Real students,Recall (%)
0,Third,47,19.0
1,Second Lower,257,77.0
2,Second Upper,264,78.0
3,First,42,64.0


In [16]:
import joblib
from google.colab import files

joblib.dump(clf4, "degree_class_model.pkl")
files.download("degree_class_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>